In [1]:
import json
from pathlib import Path
import pandas as pd

In [6]:
# Caminhos dos diretórios
PASTA_CSV = Path(
    "/home/yan/Documents/GitHub/competitive-poke-ai321/Codes/data _preprocessing_and _acquisition/documents"
)
PASTA_DOCS = Path(
    "teste"
)
PASTA_DOCS.mkdir(parents=True, exist_ok=True)

# Funções de normalização
def normalize_key(key):

    # Mantém apenas caracteres alfanuméricos (remove parênteses, %, +, sublinhados e espaços)
    return "".join(c for c in str(key).strip().lower() if c.isalnum())


def buscar_val(row, possiveis_nomes, padrao=""):

    # Busca um valor na linha verificando se o termo procurado está contido no nome da coluna
    row_norm = {normalize_key(k): v for k, v in row.items()}
    for nome in possiveis_nomes:
        nome_norm = normalize_key(nome)
        for k_norm, val in row_norm.items():
            if nome_norm in k_norm and pd.notna(val):
                val_str = str(val).strip()
                if val_str:
                    return val_str
    return padrao


def limpar_texto(texto):
    
    # Limpa caracteres corrompidos de encoding
    if not texto:
        return ""
    return str(texto).replace("\ufffd", "x").replace("×", "x").strip()



# Conversores Específicos por Arquivo
def converter_abilities(caminho):
    df = pd.read_csv(caminho).fillna("")
    documentos = []
    for _, row in df.iterrows():
        habilidade = buscar_val(
            row, ["Habilidade", "Ability", "Name", "Nome", "ability_name"]
        )
        efeito = limpar_texto(
            buscar_val(
                row,
                [
                    "Efeito",
                    "Effect",
                    "Description",
                    "Descricao",
                    "Descrição",
                ],
            )
        )

        if not habilidade:
            continue

        texto = f"Pokémon Ability: {habilidade}. Effect: {efeito}"
        documentos.append({"text": texto})

    caminho_saida = PASTA_DOCS / "31_pokemon_abilities_formatted.json"
    caminho_saida.write_text(
        json.dumps(documentos, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"Convertido: {caminho_saida.name} ({len(documentos)} habilidades salvas)"
    )


def converter_natures(caminho):
    df = pd.read_csv(caminho).fillna("")
    documentos = []

    inc_keys = [
        "increases",
        "increased",
        "statup",
        "aumenta",
        "plus",
        "boost",
        "stat_up",
    ]
    dec_keys = [
        "decreases",
        "decreased",
        "statdown",
        "diminui",
        "minus",
        "penalty",
        "stat_down",
    ]

    for _, row in df.iterrows():
        nature = buscar_val(row, ["nature", "natureza", "name", "nome"])
        increases = buscar_val(row, inc_keys)
        decreases = buscar_val(row, dec_keys)

        if not nature:
            continue

        val_inc_clean = increases.lower()
        val_dec_clean = decreases.lower()

        # Verifica se é uma Nature neutra (ex: "Nenhum (Neutra)", "none", "-", etc.)
        is_neutral = False
        if not increases or not decreases:
            is_neutral = True
        elif val_inc_clean == val_dec_clean:
            is_neutral = True
        elif any(
            w in val_inc_clean
            for w in ["nenhum", "neutral", "none", "-", "neutra"]
        ):
            is_neutral = True

        if not is_neutral:
            texto = f"Pokémon Nature: {nature}. Increases {increases} by 10% and decreases {decreases} by 10%."
        else:
            texto = (
                f"Pokémon Nature: {nature}. Neutral nature with no stat"
                " modifications."
            )

        documentos.append({"text": texto})

    caminho_saida = PASTA_DOCS / "22_pokemon_natures_formatted.json"
    caminho_saida.write_text(
        json.dumps(documentos, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"Convertido: {caminho_saida.name} ({len(documentos)} naturezas salvas)"
    )


def converter_moves(caminho):
    df = pd.read_csv(caminho).fillna("")
    documentos = []
    for _, row in df.iterrows():
        move = buscar_val(
            row,
            [
                "Nome_Ataque",
                "NomeAtaque",
                "Move",
                "Golpe",
                "Name",
                "Nome",
                "move_name",
            ],
        )
        tipo = buscar_val(row, ["Tipo", "Type", "move_type"])
        categoria = buscar_val(
            row, ["Categoria", "Category", "Class", "move_category"]
        )
        power = buscar_val(
            row, ["Poder", "Power", "Base Power", "base_power"], padrao="0"
        )
        accuracy = buscar_val(
            row, ["Precisão", "Precisao", "Accuracy", "acc"], padrao="100%"
        )
        pp = buscar_val(row, ["PP", "pp"], padrao="--")
        effect = limpar_texto(
            buscar_val(
                row,
                [
                    "Descrição",
                    "Descricao",
                    "Efeito",
                    "Effect",
                    "Description",
                ],
            )
        )

        if not move:
            continue

        texto = f"Pokémon Move: {move}. Type: {tipo}. Category: {categoria}. Base Power: {power}. Accuracy: {accuracy}. PP: {pp}."
        if effect:
            texto += f" Effect: {effect}"

        documentos.append({"text": texto})

    caminho_saida = PASTA_DOCS / "35_pokemon_moves_formatted.json"
    caminho_saida.write_text(
        json.dumps(documentos, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Convertido: {caminho_saida.name} ({len(documentos)} golpes salvos)")


def converter_types(caminho):
    df = pd.read_csv(caminho).fillna("")
    documentos = []
    for _, row in df.iterrows():
        type_name = str(
            row.get("type", row.get("Type", row.iloc[0]))
        ).strip().capitalize()
        if not type_name or type_name.isdigit():
            continue

        row_norm = {normalize_key(k): str(v).strip() for k, v in row.items()}

        double_to = str(row_norm.get("doubledamageto", "")).replace("|", ", ")
        half_to = str(row_norm.get("halfdamageto", "")).replace("|", ", ")
        no_to = str(row_norm.get("nodamageto", "")).replace("|", ", ")
        double_from = str(row_norm.get("doubledamagefrom", "")).replace("|", ", ")
        half_from = str(row_norm.get("halfdamagefrom", "")).replace("|", ", ")

        parts = []
        if double_to:
            parts.append(
                f"Deals 2x (super-effective) damage to {double_to}"
            )
        if half_to:
            parts.append(
                f"Deals 0.5x (not very effective) damage to {half_to}"
            )
        if no_to:
            parts.append(f"Deals 0x (no effect) damage to {no_to}")
        if double_from:
            parts.append(
                f"Takes 2x damage from (weak against) {double_from}"
            )
        if half_from:
            parts.append(f"Takes 0.5x damage from (resists) {half_from}")

        texto = f"Type Effectiveness for {type_name} type: " + ". ".join(parts) + "."
        documentos.append({"text": texto})

    caminho_saida = PASTA_DOCS / "47_pokemon_types_formatted.json"
    caminho_saida.write_text(
        json.dumps(documentos, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"Convertido: {caminho_saida.name} ({len(documentos)} tipos salvos)"
    )


def converter_type_stats(caminho):
    df = pd.read_csv(caminho).fillna("")
    documentos = []
    for _, row in df.iterrows():
        type_name = str(row.iloc[0]).strip().capitalize()
        if not type_name or type_name.isdigit():
            continue

        sup_eff = buscar_val(row, ["Super Effective Against"], "0")
        strong_def = buscar_val(row, ["Strong Defense Against"], "0")
        immune = buscar_val(row, ["Immune To"], "0")
        no_eff = buscar_val(row, ["No effect Against"], "0")
        not_very_eff = buscar_val(row, ["Not very Effective Against"], "0")
        weak_def = buscar_val(row, ["Weak Defence Against"], "0")

        texto = (
            f"Pokémon Type Stat Summary for {type_name}: Super effective"
            f" against {sup_eff} types. Resists (strong defense against)"
            f" {strong_def} types. Immune to {immune} types. Deals no effect"
            f" against {no_eff} types. Not very effective against"
            f" {not_very_eff} types. Weak against {weak_def} types."
        )

        documentos.append({"text": texto})

    caminho_saida = PASTA_DOCS / "56_pokemon_type_stats_formatted.json"
    caminho_saida.write_text(
        json.dumps(documentos, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"Convertido: {caminho_saida.name} ({len(documentos)} resumos de tipos salvos)"
    )


def converter_items(caminho):
    df = pd.read_csv(caminho).fillna("")
    documentos = []
    for _, row in df.iterrows():
        item = buscar_val(row, ["Item", "Name", "Nome", "item_name"])
        efeito = limpar_texto(
            buscar_val(
                row,
                [
                    "Descrição",
                    "Descricao",
                    "Efeito",
                    "Effect",
                    "Description",
                ],
            )
        )

        if not item:
            continue

        texto = f"Pokémon Item: {item}. Effect: {efeito}"
        documentos.append({"text": texto})

    caminho_saida = PASTA_DOCS / "74_pokemon_items_formatted.json"
    caminho_saida.write_text(
        json.dumps(documentos, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Convertido: {caminho_saida.name} ({len(documentos)} itens salvos)")


def converter_pokedex(caminho):
    df = pd.read_csv(caminho).fillna("")
    documentos = []
    for _, row in df.iterrows():
        name = buscar_val(row, ["Pokemon", "Name", "Nome", "species"])
        type1 = buscar_val(row, ["Tipo 1", "Type 1", "Tipo1", "Type1"])
        type2 = buscar_val(row, ["Tipo 2", "Type 2", "Tipo2", "Type2"])
        types = f"{type1}/{type2}" if type2 else type1

        hp = buscar_val(row, ["HP", "hp"], "0")
        atk = buscar_val(row, ["Attack", "Atk", "Ataque"], "0")
        defn = buscar_val(row, ["Defense", "Def", "Defesa"], "0")
        spa = buscar_val(
            row, ["Sp. Atk", "SpA", "Special Attack", "Ataque Especial"], "0"
        )
        spd = buscar_val(
            row, ["Sp. Def", "SpD", "Special Defense", "Defesa Especial"], "0"
        )
        spe = buscar_val(row, ["Speed", "Spe", "Velocidade"], "0")

        if not name:
            continue

        texto = f"Pokémon: {name}. Type: {types}. Base Stats: HP {hp}, Atk {atk}, Def {defn}, SpA {spa}, SpD {spd}, Spe {spe}."
        documentos.append({"text": texto})

    caminho_saida = PASTA_DOCS / "81_pokemon_pokedex_formatted.json"
    caminho_saida.write_text(
        json.dumps(documentos, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"Convertido: {caminho_saida.name} ({len(documentos)} pokémon salvos)"
    )


if __name__ == "__main__":
    conversores = {
        "22_pokemon_natures.csv": converter_natures,
        "31_pokemon_abilities.csv": converter_abilities,
        "35_pokemon_moves.csv": converter_moves,
        "47_pokemon_types.csv": converter_types,
        "56_pokemon_type_stats.csv": converter_type_stats,
        "74_pokemon_items.csv": converter_items,
        "81_pokemon_pokedex.csv": converter_pokedex,
    }

    print("Iniciando a conversão dos arquivos CSV...")
    for arquivo, funcao in conversores.items():
        caminho_arquivo = PASTA_CSV / arquivo
        if caminho_arquivo.exists():
            funcao(caminho_arquivo)
        else:
            print(f"Aviso: Arquivo '{arquivo}' não encontrado em {PASTA_CSV}")

Iniciando a conversão dos arquivos CSV...
Convertido: 22_pokemon_natures_formatted.json (25 naturezas salvas)
Convertido: 31_pokemon_abilities_formatted.json (314 habilidades salvas)
Convertido: 35_pokemon_moves_formatted.json (937 golpes salvos)
Convertido: 47_pokemon_types_formatted.json (18 tipos salvos)
Convertido: 56_pokemon_type_stats_formatted.json (18 resumos de tipos salvos)
Convertido: 74_pokemon_items_formatted.json (141 itens salvos)
Convertido: 81_pokemon_pokedex_formatted.json (1215 pokémon salvos)
